# Configuration

In [2]:
import os 
import warnings
warnings.filterwarnings('ignore')
if True ^ os.getcwd().endswith('hte-and-targeting'):
    os.chdir('..')

In [3]:
import pandas as pd 
import numpy as np

In [4]:
from scipy.stats import norm, invgamma
# estimators
# from econml.grf import CausalForest
from statsmodels.regression.linear_model import OLS, WLS

from scipy.integrate import quad 
from scipy.stats import ks_2samp
from scipy.optimize import fmin, minimize

In [5]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm
import plotly.express as px
import plotly.graph_objects as go

plt.rcParams['text.usetex'] = False

In [6]:
from core.variables import * 
from core.pricing_estimators import *
from core.dgp import *
from core.experiments import *
from core.visualization import *

# Experiments

In [7]:
class PricingMNBValueCorrection(PricingPlugIn):
    def __init__(self, cov_dim: int, plugin_estmr: PricingPlugIn):
        # attributes
        self.cov_dim = cov_dim

        # place holders
        self.n_bootstraps = None  # number of bootstrap samples
        self.plugin_estmr = plugin_estmr  # plugin estimator
        self.boot_targ_val_list = []

        self.m_list = []
        self.m_boot_util_params = []

    def fit(
        self, covariates: np.ndarray, prices: np.ndarray, outcomes: np.ndarray, n_bootstraps: int = 100, q: float = 0.9, max_j: int = 20
    ):
        # fill in placeholders
        self.n_bootstraps = n_bootstraps

        # calculate the sample sizes 
        sample_size = covariates.shape[0] 
        self.m_list = np.array([int(q**j * sample_size) for j in range(max_j)])

        for m in self.m_list:
            boot_util_params = np.zeros((n_bootstraps, 1 + 2 * self.cov_dim))
            # bootstrap sample indices, shape (num_bootstraps, m)
            boot_index_arr = np.random.choice(
                np.arange(covariates.shape[0]), size=(self.n_bootstraps, m), replace=True
            )  
            
            boot_cov_arr = covariates[boot_index_arr]  # (n_bootstraps, m, cov_dim)
            boot_outcome_arr = outcomes[boot_index_arr]  # (n_bootstraps, m)
            boot_price_arr = prices[boot_index_arr]  # (n_bootstraps, m)

            for boot_id in range(self.n_bootstraps):
                try:
                    model = self.logistic_regression(
                        covariates=boot_cov_arr[boot_id], prices=boot_price_arr[boot_id], outcomes=boot_outcome_arr[boot_id]
                    )
                    boot_util_params[boot_id, 0] = model.intercept_  # (1, )
                    boot_util_params[boot_id, 1:] = model.coef_.flatten()  # (2 * cov_dim, )
                except:
                    continue

            # drop zero rows
            boot_util_params = boot_util_params[~np.all(boot_util_params == 0, axis=1)]

            self.m_boot_util_params.append(boot_util_params)

        return self
    
    def estimate_targeting_value(
        self, covariates: np.ndarray, delta: float = 0.99
    ) -> np.ndarray:
        dstn_list = [None] * len(self.m_list)
        for m_id in range(len(self.m_list)):
            # optimize targeting for each bootstrap sample
            boot_targ_val_list = []
            for boot_id in range(self.m_boot_util_params[m_id].shape[0]):
                opt_result = self.optimize(
                    covariates=covariates, delta=delta, 
                    util_params=self.m_boot_util_params[m_id][boot_id]
                )
                if opt_result.success:
                    boot_targ_val_list.append(-opt_result.fun)
            dstn_list[m_id] = boot_targ_val_list

        # choose the best m
        discp_list = [None] * (len(self.m_list) - 1)
        for idx, (prev_dstn, current_dstn) in enumerate(zip(dstn_list[:-1], dstn_list[1:])):
            try:
                ks_stat, _ = ks_2samp(prev_dstn, current_dstn)
            except:
                ks_stat = np.inf
            discp_list[idx] = ks_stat

        min_discp_idx = np.argmin(discp_list)
        boot_avg = np.mean(dstn_list[min_discp_idx])

        # calculate plugin estimate
        plugin_estimate = self.plugin_estmr.estimate_targeting_value(
            covariates=covariates, delta=delta
        )

        # calculate the corrected treatment effect estimate
        return 2 * plugin_estimate - boot_avg

In [11]:
# parameters
train_size = 1000
test_size = 10
cov_dim = 2

price_lb, price_ub, price_diff = 1, 10, 1

# initialize the data generating process
dgp = PersonalizedPricingDGP(cov_dim=cov_dim)

cov_arr, price_arr, outcome_arr = dgp.generate_training_data(
    sample_size=train_size, price_lb=price_lb, price_ub=price_ub, price_diff=price_diff
)
test_cov_arr = dgp.generate_testing_data(sample_size=test_size)

# train estimators 
# plugin estimator
plugin_estmr = PricingPlugIn(cov_dim=cov_dim).fit(covariates=cov_arr, prices=price_arr, outcomes=outcome_arr)

# value correction estimator
vc_estmr = PricingValueCorrection(cov_dim=cov_dim, plugin_estmr=plugin_estmr).fit(
    covariates=cov_arr, prices=price_arr, outcomes=outcome_arr, n_bootstraps=200
)
mnb_vc_estmr = PricingMNBValueCorrection(cov_dim=cov_dim, plugin_estmr=plugin_estmr).fit(
    covariates=cov_arr, prices=price_arr, outcomes=outcome_arr, n_bootstraps=200, 
    q=0.75, max_j=20
)

In [12]:
def evaluate_pricing_policy(
    covariates: np.ndarray, price: float, dgp: PersonalizedPricingDGP, delta: float = 0.99
) -> float:
    """
    Evaluate the pricing policy

    Params:
    -------
    covariates: np.ndarray, individual characteristics
    price: float, price
    dgp: PersonalizedPricingDGP, data generating process
    delta: float, discount factor

    Returns:
    -------
    float: expected profit
    """

    logit = np.exp(covariates @ dgp.util_const_map + (price * covariates) @ dgp.util_price_map)
    purchase_prob = logit / (1 + logit)

    return np.mean(price * purchase_prob / (1 - delta * purchase_prob))

In [13]:
# evalute test set
# plugin estimator and its performance
plugin_targ_est = plugin_estmr.estimate_targeting_value(covariates=test_cov_arr)
true_plugin_targ_val = evaluate_pricing_policy(covariates=test_cov_arr, price=plugin_estmr.get_targeting_policy(test_cov_arr, delta=0.99), dgp=dgp)
vc_targ_est = vc_estmr.estimate_targeting_value(covariates=test_cov_arr)
mnb_vc_targ_est = mnb_vc_estmr.estimate_targeting_value(covariates=test_cov_arr)

print(f"Plugin targeting estimate: {plugin_targ_est:.4f}")
print(f"Value correction targeting estimate: {vc_targ_est:.4f}")
print(f"MNB value correction targeting estimate: {mnb_vc_targ_est:.4f}")

print(f"\nActual value of plugin targeting: {true_plugin_targ_val:.4f}")
print(f"Winner's Curse of plugin estimate: {plugin_targ_est - true_plugin_targ_val:.4f}")
print(f"Winner's Curse of value correction estimate: {vc_targ_est - true_plugin_targ_val:.4f}")
print(f"Winner's Curse of MNB value correction estimate: {mnb_vc_targ_est - true_plugin_targ_val:.4f}")

Plugin targeting estimate: 8.5378
Value correction targeting estimate: 8.0669
MNB value correction targeting estimate: 7.7246

Actual value of plugin targeting: 6.4433
Winner's Curse of plugin estimate: 2.0945
Winner's Curse of value correction estimate: 1.6236
Winner's Curse of MNB value correction estimate: 1.2813


# Experiments

In [16]:
def single_pricing_experiment(
    dgp: PersonalizedPricingDGP,
    sample_size: int, 
    targeting_params: dict, 
    estimators_dict: dict, 
    test_data: np.ndarray, 
    save_estimator: bool = False, 
) -> dict:
    """  
    Run a single experiment

    Params:
    -------
    dgp: DGP1, the data generating process
    sample_size: int, the number of training samples
    estimators_dict: dict, dictionary containing the estimators and their parameters
    test_data: np.ndarray, the targeting individuals
    save_estimator: bool, whether to save the estimator
    
    Returns:
    --------
    dict: Dictionary containing the following keys:
        - best_targ_val: float, the best targeting value
        - plugin_targ_est: float, the plugin targeting estimate
        - true_plugin_targ_val: float, the actual value of the plugin targeting decision
        - param_targ_est: float, the parametric targeting estimate
        - bayes_param_targ_est: float, the bayesian parametric targeting estimate
        - boot_correction_target_est: float, the bootstrap correction targeting estimate
    """
    # initialize result dictionary
    result_dict = {name: None for name in estimators_dict.keys()}

    # generate training data and targeting individuals
    cov_arr, price_arr, outcome_arr = dgp.generate_training_data(
        sample_size=sample_size, price_lb=price_lb, price_ub=price_ub, price_diff=price_diff
    )  # generate training data

    # define and train estimators
    for estimator_name, estimator_dict in estimators_dict.items():
        estimator = estimator_dict['estimator'](
            cov_dim=dgp.cov_dim
        ).fit(covariates=cov_arr, prices=price_arr, outcomes=outcome_arr, **estimator_dict['train_params'])
        targ_est = estimator.estimate_targeting_value(
            X=test_data, **estimator_dict['targeting_params'], delta=targeting_params['delta']
        )
        if save_estimator:
            result_dict[estimator_name] = {
                'estimator': estimator, 'targ_est': targ_est
            }
        else:
            result_dict[estimator_name] = {'targ_est': targ_est}

        if estimator_name == 'plugin':
            result_dict[estimator_name]['act_targ_val'] = evaluate_pricing_policy(
                covariates=test_data, price=estimator.get_targeting_policy(test_data, delta=targeting_params['delta']), dgp=dgp
            )

    return result_dict


In [19]:
# parameters
dgp_params = {
    'train_size': 1000,  # number of training samples
    'cov_dim': 2  # number of covariates
}
targeting_params = {
    'size': 10,  # number of arriving customers
    'delta': 0.99,  # discount factor
}
estimators_dict = {
    'plugin': {
        'estimator': PricingPlugIn,
        'train_params': {},
        'targeting_params': {}
    },
    'value_correction': {
        'estimator': PricingValueCorrection,
        'train_params': {'n_bootstraps': 200},
        'targeting_params': {}
    },
    'mnb_value_correction': {
        'estimator': PricingMNBValueCorrection,
        'train_params': {'n_bootstraps': 200, 'q': 0.75, 'max_j': 20},
        'targeting_params': {}
    }
}

In [21]:
result_dict = single_pricing_experiment(
    dgp=dgp, sample_size=dgp_params['train_size'], targeting_params=targeting_params, 
    estimators_dict=estimators_dict, test_data=test_cov_arr, save_estimator=True
)

TypeError: generate_training_data() missing 3 required positional arguments: 'price_lb', 'price_ub', and 'price_diff'